# case study group 23

> You are an employee of a fictitious company “106”, which sells gearshift systems to car manufacturers. In order to improve the product of the automatic gearshift system “K3AG1”, your company plans to analyze the damage cases of the last years. The automatic gearshift “K3AG1” is a standard component of the car brand “OEM1” and can be selected by the customer as an equipment feature of the car types “Type11” or “Type12”. From the management of your department you get the task to analyze the production data and parts lists of the customer and your suppliers. Key performance indicators such as service life, mileage, failure rates and usage data are of great importance.

## Contents

- [1. Importing the data](#1-importing-the-data)
- [2. Data preparation](#2-data-preparation)
- [3. Creation of the final dataset](#3-creation-of-the-final-dataset)
- [4. Evaluation](#4-evaluation)
- [5. Result](#5-result)

## 1. Importing the data

Import package and help function：

In [65]:
from typing import List
import pandas as pd
import numpy as np

def read_csv_auto(path: str) -> pd.DataFrame:
    """Read CSV with automatic delimiter detection and robust date parsing off ."""
    df = pd.read_csv(path, sep=None, engine="python", dtype=str)
    # Drop typical unnamed index columns, if any
    df = df.loc[:, ~df.columns.str.contains(r"^Unnamed", case=False)]
    # Strip quotes/spaces from column names
    df.columns = df.columns.str.strip().str.replace('"', '', regex=False)
    # Strip surrounding quotes/spaces from string cells
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].map(lambda x: x.strip().strip('"') if isinstance(x, str) else x)
    return df

Path to import data：

In [66]:

# Vehicles (baseline)
path_typ11 = "./data/Fahrzeug/Fahrzeuge_OEM1_Typ11.csv"
path_typ12 = "./data/Fahrzeug/Fahrzeuge_OEM1_Typ12.csv"

# Parts per vehicle
path_parts11 = "./data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ11.csv"
path_parts12 = "./data/Fahrzeug/Bestandteile_Fahrzeuge_OEM1_Typ12.csv"

# Component (K3AG1)
path_k3ag1   = "./data/Komponente/Komponente_K3AG1.csv"

# Component (K3SG1)
path_k3sg1 = "./data/Komponente/Komponente_K3SG1.csv"


Import data:

In [67]:
df_typ11  = read_csv_auto(path_typ11)
df_typ12  = read_csv_auto(path_typ12)
df_parts11 = read_csv_auto(path_parts11)
df_parts12 = read_csv_auto(path_parts12)
df_k3ag1  = read_csv_auto(path_k3ag1)
df_k3sg1 = read_csv_auto(path_k3sg1)

## 2. Data preparation

helper function：

In [68]:
def coalesce_first(df: pd.DataFrame, candidates: List[str], new_name: str) -> pd.DataFrame:
    """
    Create/overwrite df[new_name] with the first existing, non-null column among candidates.
    If none exist, the column is created with NaN.
    """
    values = None
    for c in candidates:
        if c in df.columns:
            values = df[c] if values is None else values.fillna(df[c])
    df[new_name] = values if values is not None else np.nan
    return df

process vehicle data:

In [69]:
# Tag vehicle type for provenance

df_typ11["vehicle_type"] = "Typ11"
df_typ12["vehicle_type"] = "Typ12"

# Baseline vehicles: stack, keep all columns; align by column name
df_vehicles_base = pd.concat([df_typ11, df_typ12], ignore_index=True, sort=False)

# Parts: unify the two parts tables and keep the shared columns
df_parts_all = pd.concat([df_parts11, df_parts12], ignore_index=True, sort=False)

df_parts_all["gearbox_type"] = df_parts_all["ID_Schaltung"].str.split("-").str[0]

process component data:

In [70]:
df_k3 = pd.concat([df_k3ag1, df_k3sg1], axis = 0, ignore_index=True)


df_k3 = coalesce_first(
    df_k3,
    ["ID_Schaltung", "ID_Schaltung.x", "ID_Schaltung.y"],
    new_name="ID_Schaltung"
)   

df_k3 = coalesce_first(
    df_k3,
    ["Produktionsdatum", "Produktionsdatum.x", "Produktionsdatum.y"],
    "gearbox_production_date"
)
df_k3 = coalesce_first(
    df_k3,
    ["Herstellernummer", "Herstellernummer.x", "Herstellernummer.y"],
    "gearbox_manufacturer_id"
)
df_k3 = coalesce_first(
    df_k3,
    ["Werksnummer", "Werksnummer.x", "Werksnummer.y"],
    "gearbox_plant_id"
)
df_k3 = coalesce_first(
    df_k3,
    ["Fehlerhaft", "Fehlerhaft.x", "Fehlerhaft.y"],
    "gearbox_defective_flag"
)
df_k3 = coalesce_first(
    df_k3,
    ["Fehlerhaft_Datum", "Fehlerhaft_Datum.x", "Fehlerhaft_Datum.y"],
    "gearbox_defective_date"
)
df_k3 = coalesce_first(
    df_k3,
    ["Fehlerhaft_Fahrleistung", "Fehlerhaft_Fahrleistung.x", "Fehlerhaft_Fahrleistung.y"],
    "gearbox_defective_mileage"
)

# Keep only the columns we standardized plus the join key
keep_cols_k3 = [
    "ID_Schaltung",
    "gearbox_production_date",
    "gearbox_manufacturer_id",
    "gearbox_plant_id",
    "gearbox_defective_flag",
    "gearbox_defective_date",
    "gearbox_defective_mileage",
]

keep_cols_k3 = [c for c in keep_cols_k3 if c in df_k3.columns]
df_k3 = df_k3[keep_cols_k3].drop_duplicates()

Found that the mileage of some defective cars is negative, which is likely erroneous data, so all negative mileage values are converted to 0.

In [71]:
df["gearbox_defective_mileage"] = pd.to_numeric(
    df["gearbox_defective_mileage"], errors="coerce"
)

df["gearbox_defective_mileage"] = df["gearbox_defective_mileage"].clip(lower=0)

## 3. Creation of the final dataset

Merge data:

In [72]:
# Merge with parts (by vehicle)
df_merged = df_vehicles_base.merge(
    df_parts_all,
    how="left",
    on="ID_Fahrzeug",
    suffixes=("", "_parts"),
    validate="m:1"
)

# Merge with K3AG1/K3SG1 component (by transmission ID from parts)
df_merged = df_merged.merge(
    df_k3,
    how="left",
    on="ID_Schaltung",
    validate="m:1"
)



rename and keep useful data:

In [73]:
rename_map = {
    "ID_Fahrzeug": "vehicle_id",
    "ID_Schaltung": "gearbox_id",
    "Produktionsdatum": "vehicle_production_date",
    "Herstellernummer": "vehicle_manufacturer_id",
    "Werksnummer": "vehicle_plant_id",
    "Fehlerhaft": "vehicle_defective_flag",
    "Fehlerhaft_Datum": "vehicle_defective_date",
    "Fehlerhaft_Fahrleistung": "vehicle_defective_mileage",
}

df_merged = df_merged.rename(columns=rename_map)

date_like_cols = [c for c in [
    "registration_date",
    "gearbox_production_date",
    "gearbox_defective_date",
] if c in df_merged.columns]

for c in date_like_cols:
    try:
        parsed = pd.to_datetime(df_merged[c], errors="coerce")
        # Keep ISO-8601 string for portability
        df_merged[c] = parsed.dt.strftime("%Y-%m-%d")
    except Exception:
        # Leave as-is on any parsing issues
        pass

df_keeps = df_merged[[ 
    'vehicle_id', 
    'vehicle_type',

    'gearbox_id', 
    'gearbox_type',
    'gearbox_manufacturer_id',
    'gearbox_plant_id',
    'gearbox_production_date', 
    'gearbox_defective_flag',
    'gearbox_defective_date', 
    'gearbox_defective_mileage']]


export data：

In [74]:
# output directory
output_path = "final_dataset_group_23.csv"

# Write CSV 
df_keeps.to_csv(output_path, index=False, encoding="utf-8")

## 4. Evaluation

The dataset’s sample data is as follows：

In [75]:
df_keeps.head()

,vehicle_id,vehicle_type,gearbox_id,gearbox_type,gearbox_manufacturer_id,gearbox_plant_id,gearbox_production_date,gearbox_defective_flag,gearbox_defective_date,gearbox_defective_mileage
0,11-1-11-1,Typ11,K3SG1-105-1051-32,K3SG1,105,1051,2008-11-13,0,NaN,0
1,11-1-11-2,Typ11,K3SG1-105-1051-141,K3SG1,105,1051,2008-11-13,0,NaN,0
2,11-1-11-3,Typ11,K3SG1-105-1051-106,K3SG1,105,1051,2008-11-13,0,NaN,0
3,11-1-11-4,Typ11,K3SG1-105-1051-21,K3SG1,105,1051,2008-11-13,0,NaN,0
4,11-1-11-5,Typ11,K3SG1-105-1051-59,K3SG1,105,1051,2008-11-13,0,NaN,0


The dataset fields are as follows:

- vehicle_id – Unique vehicle ID

- vehicle_type – Vehicle model/type (e.g., Type11, Type12)

- gearbox_id – Unique gearbox ID

- gearbox_type – Gearbox model (K3AG1, K3SG1)

-  gearbox_manufacturer_id – Gearbox manufacturer code (106 = us, others = competitors)

- gearbox_plant_id – Production plant code

- gearbox_production_date – Gearbox production date (YYYY-MM-DD)

- gearbox_defective_flag – Failure flag (1 = defective, 0 = no defect)

- gearbox_defective_date – Failure date (if any)

- gearbox_defective_mileage – Mileage at failure (km)

Overview of the data：

In [80]:
# --- Ensure date columns are datetime ---
for col in ["gearbox_production_date", "gearbox_defective_date"]:
    if col in df_keeps.columns and not pd.api.types.is_datetime64_any_dtype(df_keeps[col]):
        df_keeps[col] = pd.to_datetime(df_keeps[col], errors="coerce")

print("\n=== Overview of the data ===")

# 1) Row count
n_rows = len(df_keeps)

# 2) Distinct values
unique_vehicle_type = sorted(df_keeps["vehicle_type"].dropna().astype(str).unique()) \
    if "vehicle_type" in df_keeps.columns else []
unique_manu = sorted(df_keeps["gearbox_manufacturer_id"].dropna().astype(str).unique()) \
    if "gearbox_manufacturer_id" in df_keeps.columns else []
unique_plant = sorted(df_keeps["gearbox_plant_id"].dropna().astype(str).unique()) \
    if "gearbox_plant_id" in df_keeps.columns else []

# 3) Date ranges
prod_min = df_keeps["gearbox_production_date"].min() if "gearbox_production_date" in df_keeps.columns else pd.NaT
prod_max = df_keeps["gearbox_production_date"].max() if "gearbox_production_date" in df_keeps.columns else pd.NaT
def_min = df_keeps["gearbox_defective_date"].min() if "gearbox_defective_date" in df_keeps.columns else pd.NaT
def_max = df_keeps["gearbox_defective_date"].max() if "gearbox_defective_date" in df_keeps.columns else pd.NaT

# 4) Defective flag counts
flag_counts = df_keeps["gearbox_defective_flag"].value_counts(dropna=False).sort_index() \
    if "gearbox_defective_flag" in df_keeps.columns else pd.Series(dtype="int64")

# 5) Mileage range
mileage_min = df_keeps["gearbox_defective_mileage"].min(skipna=True) if "gearbox_defective_mileage" in df_keeps.columns else None
mileage_max = df_keeps["gearbox_defective_mileage"].max(skipna=True) if "gearbox_defective_mileage" in df_keeps.columns else None

# --- Print results ---
print(f"Rows: {n_rows:,}")

print("\nDistinct values:")
print(f"- vehicle_type ({len(unique_vehicle_type)}): {unique_vehicle_type}")
print(f"- gearbox_manufacturer_id ({len(unique_manu)}): {unique_manu}")
print(f"- gearbox_plant_id ({len(unique_plant)}): {unique_plant}")

print("\nDate ranges:")
prod_min_str = prod_min.date() if pd.notna(prod_min) else "NA"
prod_max_str = prod_max.date() if pd.notna(prod_max) else "NA"
def_min_str = def_min.date() if pd.notna(def_min) else "NA"
def_max_str = def_max.date() if pd.notna(def_max) else "NA"
print(f"- gearbox_production_date: {prod_min_str} → {prod_max_str}")
print(f"- gearbox_defective_date:  {def_min_str} → {def_max_str}")

print("\ngearbox_defective_flag counts (including NaN if present):")
for k, v in flag_counts.items():
        label = "NaN" if pd.isna(k) else int(k)
        print(f"- {label}: {v:,}")

print("\nMileage range (gearbox_defective_mileage):")

print(f"- Min: {mileage_min}, Max: {mileage_max}")



=== Overview of the data ===
Rows: 2,385,260

Distinct values:
- vehicle_type (2): ['Typ11', 'Typ12']
- gearbox_manufacturer_id (4): ['105', '106', '107', '108']
- gearbox_plant_id (4): ['1051', '1061', '1071', '1082']

Date ranges:
- gearbox_production_date: 2008-11-12 → 2016-11-15
- gearbox_defective_date:  2009-03-05 → 2018-07-21

gearbox_defective_flag counts (including NaN if present):
- 0: 2,133,266
- 1: 251,994

Mileage range (gearbox_defective_mileage):
- Min: -180, Max: 9945


Check for missing data: 

In [79]:
print("\n=== Missingness & Dtype ===")
missing = df_keeps.isna().sum().sort_values(ascending=False)
dtypes = df_keeps.dtypes.astype(str)
miss_rate = (df_keeps.isna().mean()*100).round(2)
miss_summary = pd.DataFrame({"dtype": dtypes, "missing_cnt": missing, "missing_rate_%": miss_rate}) \
    .sort_values(["missing_cnt","missing_rate_%"], ascending=False)
print(miss_summary)


=== Missingness & Dtype ===
                                    dtype  missing_cnt  missing_rate_%
gearbox_defective_date     datetime64[ns]      2133266           89.44
gearbox_defective_flag             object            0            0.00
gearbox_defective_mileage          object            0            0.00
gearbox_id                         object            0            0.00
gearbox_manufacturer_id            object            0            0.00
gearbox_plant_id                   object            0            0.00
gearbox_production_date    datetime64[ns]            0            0.00
gearbox_type                       object            0            0.00
vehicle_id                         object            0            0.00
vehicle_type                       object            0            0.00


Check for duplicate data：

In [ ]:
print("\n=== Key Integrity & Duplicates ===")
for key_col in ["vehicle_id", "gearbox_id"]:
    if key_col in df_keeps.columns:
        dup_cnt = df_keeps.duplicated(subset=[key_col]).sum()
        print(f"- Duplicates by {key_col}: {dup_cnt:,}")
print(f"- Fully duplicated rows: {df_keeps.duplicated().sum():,}")


=== Key Integrity & Duplicates ===
- Duplicates by vehicle_id: 0
- Duplicates by gearbox_id: 0
- Fully duplicated rows: 0


From the above evaluation, the dataset has been verified to be valid and consistent.

## 5. Result

To run the app, please execute the following code to install dependencies:

 ```
 pip install dash pandas plotly
 ```


Please run the command to start the app:
```
python case_study_group_23.py
```

You will be able to see the app, as shown in the figure below：

<img src="additional_files/dapp-overviw.png" width="300">


### 5.1 Market Share Analysis

First, consider the market share.

Considering the two components, **K3AG1** and **K3SG1**, the market share of each manufacturer is shown in the figure below:  

<img src="additional_files/market-k3ag1.jpg" width="600">

For our company, **Manufacturer 106**, the market share has remained relatively stable at around **10%**, which is comparatively low but consistent over time. **Manufacturer 107** holds the largest share, averaging approximately **48%**, followed by **Manufacturer 105** at about **32%**. The lowest share is recorded by **Manufacturer 106 (point data)** at only **4%**.  

When considering only **K3AG1** produced by our company, the market share distribution among manufacturers is shown below:  
Our company’s share remains stable at around **50%**, peaking at **54%** in October 2016. Competitor **Manufacturer 105** accounts for roughly **20%**, while **Manufacturer 108** holds approximately **30%**.  

<img src="additional_files/market-k3ag1.jpg" width="600">

In summary, our company’s market share in gearboxes has remained stable. Specifically, in the **K3AG1** model, our share is significantly higher than that of competitors. However, since the **K3SG1**, which we do not produce, is more widely used, our overall share in the gearbox market remains moderate.  


### 5.2 Failure Rate Analysis  

After that, observe the occurrence of failures in the gearbox components.

The figure below shows the **failure rate trends over time (2009–2016)** for gearshift systems produced by different manufacturers:  

<img src="additional_files/failure-rate.jpg" width="400">

- **Our company (Manufacturer 106, blue):**  
  The failure rate has remained consistently between **9% and 12%**, with only minor fluctuations. This reflects a **low and stable level**, demonstrating consistent product quality and reliability.  

- **Manufacturer 105 (orange):**  
  The failure rate averages around **20%**, fluctuating between **18% and 22%**. Compared to our company, the failure rate is nearly **twice as high**, indicating significantly weaker product reliability.  

- **Manufacturer 108 (light orange):**  
  The failure rate generally ranges from **8% to 12%**, similar to our company. However, it shows **greater volatility**, occasionally dropping below **9%** but also rising above **12%**. Compared to Manufacturer 106, it performs **less stably**.  


Our company (**106**) has maintained a **stable and consistently low failure rate**, significantly outperforming **Manufacturer 105** and showing comparable results to **Manufacturer 108**, but with **greater stability**. This indicates that our gearshift systems provide **strong long-term reliability and durability**, offering a clear competitive advantage.  


### 5.3 Error Frequency Analysis  

Next, compare the total number of failures with that of the competitors.

The figures below compare the **error frequency of the K3AG1 gearbox component** across different manufacturers:  

<img src="additional_files/error-frequency.jpg" width="400">

- **Left chart (Defective units):**  
  Our company (**106**) reports approximately **24,000 defective units**, while other manufacturers show close to **35,000 defective units**. Overall, our defective unit count is **significantly lower than that of competitors**.  

- **Right chart (Defect rate):**  
  Our company (**106**) has a defect rate of about **10%**, whereas other manufacturers exceed **14%**. This indicates that, at a comparable production scale, our products achieve **greater quality consistency and lower defect rates**.  

Both in terms of **absolute defective units** and **relative defect rates**, our company (**106**) outperforms competitors in the K3AG1 product line. Particularly with respect to defect rate, our performance is **below the industry average**, highlighting our competitive advantage in **product reliability and production consistency**.  

### 5.4 Survival (Life Time) Analysis 

Now, analyze the relationship between failures and component lifespan. Note that the lifespan is calculated as the failure time minus the production time of the component, which may differ from the actual usage time.

The figure below shows the **survival rate (S(t)) over time** for the K3AG1 automatic gearbox produced by different manufacturers:  

<img src="additional_files/lifetime.jpg" width="400">

- **Our company (Manufacturer 106, blue):**  
  The survival rate declines slowly from the initial **100%**, remaining above **90%** throughout the observation period (around 600 days). This indicates **high product lifetime and reliability**.  

- **Manufacturer 105 (light brown):**  
  A steep drop is observed early in the lifecycle, with the survival rate falling rapidly to about **80%**. This suggests a **higher early failure rate** and significantly lower durability compared to our company.  

- **Manufacturer 108 (light yellow):**  
  The survival curve is similar to our company’s, with a gradual decline and stabilization around **90%**. However, its performance is slightly lower, showing **comparable but less consistent reliability**.  

Our company (**106**) demonstrates **superior product durability** compared to Manufacturer 105 and achieves performance on par with Manufacturer 108, but with **greater stability**. This highlights our competitive advantage in **long-term reliability and product lifespan**.  


### 5.5 Analysis of Mileage Distribution at Failure

Finally, analyze the failure mileage and compare it with that of the competitors.

The chart below shows the **failure mileage distribution** of the K3AG1 automatic gearbox across different manufacturers:  

<img src="additional_files/mileage.jpg" width="400">

- **Our company (106, blue):**  
  Failures are concentrated around **30,000 km**, with a narrow distribution. This indicates that failures occur at higher mileage, reflecting a **longer product lifetime**.  

- **Manufacturer 105 (light brown):**  
  Failures are mostly concentrated between **5,000–10,000 km**, with a relatively narrow distribution. This suggests that products tend to fail earlier in their usage cycle, showing **significantly weaker durability**.  

- **Manufacturer 108 (light yellow):**  
  Failures are more dispersed, but most occur around **30,000 km**, similar to our company. However, the distribution is wider and less stable compared to ours.  


Our company (106) clearly outperforms Manufacturer 105 in terms of failure mileage, demonstrating **longer lifetime and stronger reliability**. Compared to Manufacturer 108, our products perform at a similar mileage level but with **greater consistency and stability**, highlighting a competitive advantage in **durability and reliability**.  


When comparing the two models, **K3AG1** and **K3SG1**, it becomes evident that failures of **K3AG1** generally occur at **higher mileage levels**.  This indicates that the gearboxes produced by **our company (106)** are **more durable** compared to competing products, providing customers with **longer service life and higher reliability**.  

<img src="additional_files/mileage-all.jpg" width="400">



### 5.6 Conclusion  

The comprehensive analysis of the **K3AG1** and **K3SG1** gearboxes demonstrates that our company (**Manufacturer 106**) maintains a clear competitive edge in multiple key performance indicators:  

- **Market Share:** Although our overall share is moderate due to not producing K3SG1, in the **K3AG1 segment** our company consistently holds a dominant position with a stable share of around **50%**, significantly higher than competitors.  
- **Failure Rate:** Our products achieve a **low and stable failure rate (9–12%)**, outperforming Manufacturer 105 and showing greater stability compared to Manufacturer 108.  
- **Error Frequency:** Both the absolute number of defective units and the defect rate are **lower for 106**, highlighting superior production quality and process consistency.  
- **Lifetime (Survival Analysis):** Our gearboxes demonstrate **long-term durability**, remaining above 90% survival during the observation period, which is superior to 105 and comparable but more stable than 108.  
- **Failure Mileage:** Failures of our gearboxes occur at **higher mileage (around 30,000 km)**, indicating longer service life. In contrast, competitors such as 105 show significantly earlier failures.  

**In conclusion, Manufacturer 106 delivers gearshift systems with higher durability, reliability, and product quality compared to competitors. While the absence of K3SG1 production limits our overall market share, our strong performance in the K3AG1 model underscores our competitive advantage in both technical quality and long-term customer value.**